# Packages to Import

In [1]:
from __future__ import print_function, unicode_literals, absolute_import, division
import sys
import numpy as np
import matplotlib
matplotlib.rcParams["image.interpolation"] = 'none'
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from glob import glob
from tifffile import imread
from csbdeep.utils import Path, normalize
from csbdeep.io import save_tiff_imagej_compatible

from stardist import random_label_cmap, _draw_polygons, export_imagej_rois
from stardist.models import StarDist2D

np.random.seed(6)
lbl_cmap = random_label_cmap()

from tqdm import tqdm
import napari
import skimage as sk
import pandas as pd
import os
from aicspylibczi import CziFile

## StarDist Prediction and Object Analysis
This section is specifically for detecting and quantifying the ring-like structures observed in the oocytes

### Prepping image slices to be analyzed

In [ ]:
#fix file names replacing spaces with underscores
from Fix_File_Names import replace_spaces_with_underscores_recursive
replace_spaces_with_underscores_recursive(r"Image_Data\Oct_2025_300mm_exp")

In [10]:
na_50_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\50_mM\*.czi'))
na_300_1_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\*.czi'))
na_300_6_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\*.czi'))

In [18]:
na_50_files_names = list(map(os.path.basename,na_50_files))
na_300_1_files_names = list(map(os.path.basename,na_300_1_files))
na_300_6_files_names = list(map(os.path.basename,na_300_6_files))

In [21]:
print(na_300_1_files_names)

['Image_4_Airyscan_Processing.czi', 'Image_5_Airyscan_Processing.czi']


In [11]:
na_50_czi = list(map(CziFile,na_50_files))
na_300_1_czi = list(map(CziFile,na_300_1_files))
na_300_6_czi = list(map(CziFile,na_300_6_files))

In [36]:
#reference names
na_50_pos_z_files= sorted(glob(r'Image_Data\tomm20GFP_Oct2025\50_mm\*.czi'))
na_300_1_pos_z_files = sorted(glob(r'Image_Data\tomm20GFP_Oct2025\300_mm_1_hr\*.czi'))
na_300_6_pos_z_files = sorted(glob(r'Image_Data\tomm20GFP_Oct2025\300_mm_6_hr\*.czi'))

In [37]:
#get base name of all images
na_50_pos_z_names = list(map(os.path.basename,na_50_pos_z_files))
na_300_1_pos_z_names = list(map(os.path.basename,na_300_1_pos_z_files))
na_300_6_pos_z_names = list(map(os.path.basename,na_300_6_pos_z_files))

Get list of image positions and slices to analyze using naming pattern from extracted slices with ROIs, created tuples of the scenes and z slices to use for extracting the data subset from the original data.

In [39]:
#lists of positions and z for analysis
import re
from collections import defaultdict

def extract_and_group_scenes(file_list):
    """
    Extract (pos, z) tuples from filenames and group by source image.
    
    Args:
        file_list: List of filenames with pattern like 'airyscan_6_pos1_z_8.czi'
    
    Returns:
        List of lists, where each inner list contains (pos, z) tuples grouped by source image
    """
    # Dictionary to group tuples by source image prefix
    grouped = defaultdict(list)
    
    for filename in file_list:
        # Extract the source image prefix (everything before _pos)
        source_match = re.match(r'^([^_]*_[^_]*)_pos', filename)
        source_key = source_match.group(1) if source_match else None
        
        # Extract pos number and z number
        pos_match = re.search(r'_pos(\d+)_', filename)
        z_match = re.search(r'_z_(\d+)', filename)
        
        if pos_match and z_match and source_key:
            pos = int(pos_match.group(1))
            z = int(z_match.group(1))
            grouped[source_key].append((pos, z))
    
    # Convert to list of lists, maintaining order
    result = list(grouped.values())
    return result

# Example usage:
file_list = [
    'airyscan_6_pos1_z_8.czi',
    'airyscan_6_pos3_z_38_2.czi',
    '29_airyscan_pos1_z_26.czi',
    '29_airyscan_pos2_z_17_2.czi'
]

na_50_pos_z = extract_and_group_scenes(na_50_pos_z_names)
na_300_1_pos_z = extract_and_group_scenes(na_300_1_pos_z_names)
na_300_6_pos_z = extract_and_group_scenes(na_300_6_pos_z_names)


Found additional inconsistent naming schemes, names of data subset for image 5 from the 300 mM Na dataset had an added underscore between "pos" and the number. Had to fix this for the data extraction to run smoothly.

In [34]:
print("na_300_1_pos_z_names:")
print(na_300_1_pos_z_names)
print("\nna_300_6_pos_z_names:")
print(na_300_6_pos_z_names)

na_300_1_pos_z_names:
['airyscan_4_pos1_z_23.czi', 'airyscan_4_pos1_z_23_2.czi', 'airyscan_4_pos1_z_8.czi', 'airyscan_4_pos1_z_8_2.czi', 'airyscan_4_pos2_z_17.czi', 'airyscan_4_pos3_z_10.czi', 'airyscan_4_pos3_z_20.czi', 'airyscan_4_pos4_z_21.czi', 'airyscan_4_pos5_z_13.czi', 'airyscan_4_pos5_z_38.czi', 'airyscan_4_pos6_z_10.czi', 'airyscan_4_pos6_z_10_2.czi', 'airyscan_4_pos6_z_30.czi', 'airyscan_4_pos6_z_30_2.czi', 'airyscan_5_pos_10_z_28.czi', 'airyscan_5_pos_10_z_40.czi', 'airyscan_5_pos_10_z_52.czi', 'airyscan_5_pos_10_z_52_2.czi', 'airyscan_5_pos_2_z_41.czi', 'airyscan_5_pos_2_z_48.czi', 'airyscan_5_pos_5_z_16.czi', 'airyscan_5_pos_5_z_16_2.czi', 'airyscan_5_pos_5_z_43.czi', 'airyscan_5_pos_5_z_43_2.czi', 'airyscan_5_pos_6_z_32.czi', 'airyscan_5_pos_9_z_23.czi']

na_300_6_pos_z_names:
['airyscan_6_pos1_z_22.czi', 'airyscan_6_pos1_z_8.czi', 'airyscan_6_pos2_z_12.czi', 'airyscan_6_pos2_z_25.czi', 'airyscan_6_pos3_z_15.czi', 'airyscan_6_pos3_z_15_2.czi', 'airyscan_6_pos3_z_38.czi', 

In [ ]:
import os
import re

def rename_pos_underscore_files(directory):
    """
    Rename files with _pos_#_ pattern to _pos#_ pattern
    Example: airyscan_6_pos_1_z_8.czi -> airyscan_6_pos1_z_8.czi
    """
    for filename in os.listdir(directory):
        if filename.endswith('.czi'):
            # Replace _pos_(\d+)_ with _pos\1_
            new_filename = re.sub(r'_pos_(\d+)_', r'_pos\1_', filename)
            
            if new_filename != filename:
                old_path = os.path.join(directory, filename)
                new_path = os.path.join(directory, new_filename)
                os.rename(old_path, new_path)
                print(f"Renamed: {filename} -> {new_filename}")

# Apply to the directory with mismatched files
rename_pos_underscore_files(r'Image_Data\tomm20GFP_Oct2025\300_mm_1_hr')

In [16]:
#where to save the extracted subset
na_50_slices_path = r'Image_Data\Oct_2025_300mm_exp\50_mM\Z_slices'
na_300_1_slices_path = r'Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Z_slices'
na_300_6_slices_path = r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Z_slices'


Selected the respective images from the original czi files and saved as individual tiffs

In [30]:
for img, pos, name in zip(na_50_czi,na_50_pos_z,na_50_files_names):
    path = na_50_slices_path
    for scene, z in pos:
        slice = np.squeeze(img.read_image(S=scene-1,Z=z-1)[0])
        sk.io.imsave(os.path.join(path,name[:-4]+'_pos'+str(scene)+'_z'+str(z)+'.tiff'),slice,check_contrast=False)

In [40]:
for img, pos, name in zip(na_300_1_czi,na_300_1_pos_z,na_300_1_files_names):
    path = na_300_1_slices_path
    for scene, z in pos:
        slice = np.squeeze(img.read_image(S=scene-1,Z=z-1)[0])
        sk.io.imsave(os.path.join(path,name[:-4]+'_pos'+str(scene)+'_z'+str(z)+'.tiff'),slice,check_contrast=False)

In [41]:
for img, pos, name in zip(na_300_6_czi,na_300_6_pos_z,na_300_6_files_names):
    path = na_300_6_slices_path
    for scene, z in pos:
        slice = np.squeeze(img.read_image(S=scene-1,Z=z-1)[0])
        sk.io.imsave(os.path.join(path,name[:-4]+'_pos'+str(scene)+'_z'+str(z)+'.tiff'),slice,check_contrast=False)

Double check if there are any other files names that are not matching the naming scheme

In [ ]:
def extract_and_group_scenes_debug(file_list):
    """Debug version to see what's being extracted"""
    grouped = defaultdict(list)
    non_matching = []
    
    for filename in file_list:
        source_match = re.match(r'^([^_]*_[^_]*)_pos', filename)
        source_key = source_match.group(1) if source_match else None
        
        pos_match = re.search(r'_pos(\d+)_', filename)
        z_match = re.search(r'_z_(\d+)', filename)
        
        print(f"File: {filename}")
        print(f"  source_key: {source_key}, pos: {pos_match.group(1) if pos_match else None}, z: {z_match.group(1) if z_match else None}")
        
        if pos_match and z_match and source_key:
            pos = int(pos_match.group(1))
            z = int(z_match.group(1))
            grouped[source_key].append((pos, z))
        else:
            non_matching.append({
                'filename': filename,
                'source_key': source_key,
                'has_pos': pos_match is not None,
                'has_z': z_match is not None
            })
    
    result = list(grouped.values())
    print(f"\nFinal grouped result: {result}")
    
    if non_matching:
        print(f"\n⚠️  {len(non_matching)} files did NOT match the regex pattern:")
        for item in non_matching:
            print(f"  - {item['filename']}")
            print(f"    source_key: {item['source_key']}, has_pos: {item['has_pos']}, has_z: {item['has_z']}")
    else:
        print("\n✓ All files matched the regex pattern!")
    
    return result

# Test it
print("=" * 50)
print("Debugging na_300_1_pos_z_names:")
print("=" * 50)
na_300_1_pos_z_debug = extract_and_group_scenes_debug(na_300_1_pos_z_names)

In [ ]:
extract_and_group_scenes_debug(na_50_pos_z_names)

### Trained Custom StarDist Model to Detect Ring Structures

Notebook outlining stardist training is in [Transfer_learning_StarDist](/Transfer_learning_StarDist.ipynb)

Run custom StarDist model on data subset extracted above

In [65]:
na_50_slices_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\50_mM\Z_slices\*.tiff'))
na_300_1_slices_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Z_slices\*.tiff'))
na_300_6_slices_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Z_slices\*.tiff'))

In [66]:
na_50_slices_names = list(map(os.path.basename,na_50_slices_files))
na_300_1_slices_names = list(map(os.path.basename,na_300_1_slices_files))
na_300_6_slices_names = list(map(os.path.basename,na_300_6_slices_files))

In [67]:
na_50_slices = list(map(imread,na_50_slices_files))
na_300_1_slices = list(map(imread,na_300_1_slices_files))
na_300_6_slices = list(map(imread,na_300_6_slices_files))

In [68]:
#check image dimensions for image normalization
n_channel = 1 if na_50_slices[0].ndim == 2 else X[0].shape[-1]
axis_norm = (0,1)
if n_channel > 1:
    print("Normalizing image channels %s." % ('jointly' if axis_norm is None or 2 in axis_norm else 'independently'))

Load in custom trained model and then run on all images to get objects

In [54]:
#load in model
model = StarDist2D(None, name='2D_versatile_fluo_Rings', basedir='models')

Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.478404, nms_thresh=0.3.


In [69]:
na_50_slices_norm = [normalize(x, 1,99.8, axis=axis_norm) for x in na_50_slices]
na_300_1_slices_norm = [normalize(x, 1,99.8, axis=axis_norm) for x in na_300_1_slices]
na_300_6_slices_norm = [normalize(x, 1,99.8, axis=axis_norm) for x in na_300_6_slices]


In [70]:
#Predict labels on full images, mask the labels with the manually created masks to remove background objects
na_50_slices_labels = [model.predict_instances(x)[0] for x in tqdm(na_50_slices_norm)]
na_300_1_slices_labels = [model.predict_instances(x)[0] for x in tqdm(na_300_1_slices_norm)]
na_300_6_slices_labels = [model.predict_instances(x)[0] for x in tqdm(na_300_6_slices_norm)]

100%|██████████| 30/30 [00:12<00:00,  2.43it/s]


Check labels to see how model performed

In [61]:
viewer = napari.view_image(na_300_1_slices[15])
viewer.add_labels(na_300_1_slices_labels[15])

1932404604.py (1): `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.


<Labels layer 'Labels' at 0x1e6e3eb37c0>

In [71]:
na_50_path = r'Image_Data\Oct_2025_300mm_exp\50_mM\Ring_masks'
na_300_1_path = r'Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Ring_masks'
na_300_6_path = r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Ring_masks'

In [72]:
#save the prediction masks
for pred, name in zip(na_50_slices_labels,na_50_slices_names):
    path = na_50_path
    sk.io.imsave(os.path.join(path,name[:-4]+'_ring_masks.tiff'),pred,check_contrast=False)

for pred, name in zip(na_300_1_slices_labels,na_300_1_slices_names):
    path = na_300_1_path
    sk.io.imsave(os.path.join(path,name[:-4]+'_ring_masks.tiff'),pred,check_contrast=False)

for pred, name in zip(na_300_6_slices_labels,na_300_6_slices_names):
    path = na_300_6_path
    sk.io.imsave(os.path.join(path,name[:-4]+'_ring_masks.tiff'),pred,check_contrast=False)

In [ ]:
na_50_masks_files = sorted(glob('50_mm/Masks/*.tif'))
na_500_masks_files = sorted(glob('500_mm/Masks/*.tif'))
na_50_masks = list(map(sk.io.imread,na_50_masks_files))
na_500_masks = list(map(sk.io.imread,na_500_masks_files))

In [ ]:
all_masks = na_50_masks + na_500_masks

In [ ]:
masked_labels = [all_masks[i]*labels[i] for i in range(len(labels))]

Get measurements of all identified objects in oocytes

In [ ]:
pixel_size = [0.07,0.07]
props = ('label','axis_major_length','axis_minor_length','eccentricity','perimeter','intensity_mean','area')

In [ ]:
def get_measurements(mask,img,props,pixel_size):
    df = sk.measure.regionprops_table(mask,img,properties=props,spacing=pixel_size)
    df = pd.DataFrame.from_dict(df)
    return df

In [ ]:
def circularity_and_aspect_ratio(df):
    circularity_list = []
    aspect_ratio_list = []
    perimeters = np.asarray(df['perimeter']).astype(np.float64)
    areas = np.asarray(df['area']).astype(np.float64)
    min_diam = np.asarray(df['axis_minor_length']).astype(np.float64)
    max_diam = np.asarray(df['axis_major_length']).astype(np.float64)
    for c in range(len(perimeters)):
        circ = (4*np.pi*areas[c])/(perimeters[c]**2)
        circularity_list.append(circ)
        ar = max_diam[c]/min_diam[c]
        aspect_ratio_list.append(ar)
    aspect_ratios = pd.Series(aspect_ratio_list,name='aspect_ratio')
    circularities = pd.Series(circularity_list,name='circularity')
    merged_df = pd.concat([df,circularities,aspect_ratios],axis=1)
    return merged_df


In [ ]:
def save(save_path, img_name, mask, merged_df):
    masks_path = os.path.join(save_path,'masks')
    dataframe_path = os.path.join(save_path,'measurements')
    sk.io.imsave(os.path.join(masks_path,'masks_'+img_name[:-5]+'.tif'),mask,check_contrast=False)
    merged_df.to_csv(os.path.join(dataframe_path,'measurements_'+img_name[:-5]+'.csv'))

In [ ]:
save_path = 'Output'
all_imgs = na_50_imgs + na_500_imgs
for i in range(len(file_list)):
    img_name = os.path.basename(file_list[i])
    mask = masked_labels[i]
    img = all_imgs[i]
    df = get_measurements(mask,img,props,pixel_size)
    merged_df = circularity_and_aspect_ratio(df)
    save(save_path,img_name,mask,merged_df)

Options for plotting per image data to inspect results

In [ ]:
na_50_measurements = sorted(glob('Output/measurements/*_50mm_*.csv'))
na_500_measurements = sorted(glob('Output/measurements/*_500mm_*.csv'))

In [ ]:
na_50_name_list = [os.path.basename(file)[13:-4] for file in na_50_measurements]
na_500_name_list = [os.path.basename(file)[13:-4] for file in na_500_measurements]

In [ ]:
na_50_measurements = list(map(pd.read_csv,na_50_measurements))
na_500_measurements = list(map(pd.read_csv,na_500_measurements))


In [ ]:
df1 = na_50_measurements[3]
df2 = na_50_measurements[0]

In [ ]:
#get a look at the measurements and the names of the columns
df1.head()

In [ ]:
area_50 = [df['area'] for df in na_50_measurements]
area_500 = [df['area'] for df in na_500_measurements]
ar_50 = [df['aspect_ratio'] for df in na_50_measurements]
ar_500 = [df['aspect_ratio'] for df in na_500_measurements]
circ_50 = [df['circularity'] for df in na_50_measurements]
circ_500 = [df['circularity'] for df in na_500_measurements]
intensity_50 = [df['intensity_mean'] for df in na_50_measurements]
intensity_500 = [df['intensity_mean'] for df in na_500_measurements]

In [ ]:
fig, ax = plt.subplots(4,2,figsize=(12,10),layout='constrained')
ax[0,0].boxplot(area_50)
ax[0,0].set_title('Area 50 mM Na')
ax[0,0].set_ylim(0,6)
ax[0,1].boxplot(area_500)
ax[0,1].set_title('Area 500 mM Na')
ax[0,1].set_ylim(0,6)
ax[1,0].boxplot(ar_50)
ax[1,0].set_title('Aspect Ratio 50 mM Na')
ax[1,0].set_ylim(0.5,2.5)
ax[1,1].boxplot(ar_500)
ax[1,1].set_title('Aspect Ratio 500 mM Na')
ax[1,1].set_ylim(0.5,2.5)
ax[2,0].boxplot(circ_50)
ax[2,0].set_title('Circularity 50 mM Na')
ax[2,0].set_ylim(0.5,1.5)
ax[2,1].boxplot(circ_500)
ax[2,1].set_title('Circularity 500 mM Na')
ax[2,1].set_ylim(0.5,1.5)
ax[3,0].boxplot(intensity_50)
ax[3,0].set_title('Mean Intensity 50 mM Na')
ax[3,0].set_ylim(500,10000)
ax[3,1].boxplot(intensity_500)
ax[3,1].set_title('Mean Intensity 500 mM Na')
ax[3,1].set_ylim(500,10000)
plt.show()

## Linear mixed effects model to compare areas

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
from glob import glob
import matplotlib.pyplot as plt

In [ ]:
na_50_measurements = sorted(glob('Output/measurements/*_50mm_*.csv'))
na_500_measurements = sorted(glob('Output/measurements/*_500mm_*.csv'))

In [ ]:
na_50_measurements = list(map(pd.read_csv,na_50_measurements))
na_500_measurements = list(map(pd.read_csv,na_500_measurements))

In [ ]:
na_50_measurements[0].head()

Externally added in a column for the bio_rep and condition for setting up the mixed effects model

In [ ]:
area_df_na50 = [df.get(['area','bio_rep','condition']) for df in na_50_measurements]
area_df_na500 = [df.get(['area','bio_rep','condition']) for df in na_500_measurements]
area_df_na50_merged = pd.concat(area_df_na50).reset_index(drop=True)
area_df_na500_merged = pd.concat(area_df_na500).reset_index(drop=True)

In [ ]:
#verify concat worked
area_df_na50_merged.tail()

In [ ]:
dfs_merged = pd.concat([area_df_na50_merged,area_df_na500_merged]).reset_index(drop=True)

Testing square root and log normalization for better model performance

In [ ]:
dfs_merged['area_sqrt'] = np.sqrt(dfs_merged['area'])

In [ ]:
dfs_merged['area_log'] = np.log(dfs_merged['area'])

In [ ]:
#setting up and running mixed effects model
model = smf.mixedlm(
    "area_log ~ C(condition)",
    data=dfs_merged,
    groups=dfs_merged["bio_rep"])
mdf = model.fit()
#print(mdf.summary())

In [ ]:
residuals = mdf.resid
fitted_values = mdf.fittedvalues

In [ ]:
# Residuals plot
import statsmodels.api as sm
sm.qqplot(residuals, line='s')
plt.title("QQ-Plot of Residuals")
plt.show()

In [ ]:
print(mdf.summary())

In [ ]:
dfs_merged.to_csv(os.path.join('Output','Areas_.csv'))

### Created cropped and masked images for skeleton analysis
Skeleton Analysis outlined in [skan_analysis](/skan_analysis.ipynb)

In [ ]:
na_50_masks_files = sorted(glob('50_mm/Masks/*.tif'))
na_500_masks_files = sorted(glob('500_mm/Masks/*.tif'))
na_50_masks = list(map(sk.io.imread,na_50_masks_files))
na_500_masks = list(map(sk.io.imread,na_500_masks_files))
na_50_img_files = sorted(glob('50_mm/TIFFs/*.tiff'))
na_500_img_files = sorted(glob('500_mm/TIFFs/*.tiff'))
na_50_imgs = list(map(imread,na_50_img_files))
na_500_imgs = list(map(imread,na_500_img_files))

In [ ]:
#mask and crop images by oocyte mask bounding box
file_names = na_50_img_files + na_500_img_files
imgs = na_50_imgs + na_500_imgs
masks = na_50_masks + na_500_masks

In [ ]:
save_path = "cropped_imgs"
for i in range(len(file_names)):
    name = os.path.basename(file_names[i])
    img = imgs[i]
    mask = masks[i]
    props = sk.measure.regionprops(mask)
    minc,minr,maxc,maxr = props[0].bbox
    masked_img = mask*img
    cropped_img = masked_img[minc:maxc,minr:maxr]
    sk.io.imsave(os.path.join(save_path,name[:-5]+"_crop.tif"),cropped_img)